<a href="https://colab.research.google.com/github/ger01d/opencat-gym-brax-mujoco/blob/main/OpencatGym_Brax_Mujoco_v0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpencatGym-Brax-Mujoco
v0.1

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# RUNTIME CHECK — aborts early if not on A100
# Run All is safe: this cell raises before any compute is wasted.
# ═══════════════════════════════════════════════════════════════════════════
import subprocess

_gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

print(f"GPU detected: {_gpu}")

if "A100" not in _gpu:
    raise RuntimeError(
        f"\n  Wrong GPU: {_gpu}\n"
        "  Change runtime:  Runtime > Change runtime type > GPU: A100\n"
        "  (Requires Colab Pro+)\n"
        "  Aborting to avoid wasting compute on the wrong hardware."
    )

print("A100 confirmed.")

GPU detected: NVIDIA A100-SXM4-40GB, 40960 MiB
A100 confirmed.


In [2]:
# Install deps (Brax pulls JAX, MJX)
!pip install -q brax #flax

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.0/351.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 144.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.3/740.3 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 20.1 MB/s eta 0:00:00


In [3]:
REWARD_CONFIG = {
    # ── Forward motion (linear displacement) ──
    "fac_tracking":       1000.0,  # Reward per meter forward displacement (PyBullet used 1000)

    # ── Penalties (all zero for Phase 0 baseline) ──
    "fac_smooth_1":       0.0,     # Penalize 1st order joint jitter
    "fac_smooth_2":       0.0,     # Penalize 2nd order joint jitter
    "fac_stability":      0.0,     # Penalize body angular velocity
    "fac_orient_pitch":   0.0,     # Penalize forward/backward tilt
    "fac_orient_roll":    0.0,     # Penalize lateral tilt
    "fac_lin_vel_z":      0.0,     # Penalize vertical body velocity (anti-bounce)
    "fac_air_time":       0.0,     # Reward for feet air time (stepping)
    "air_time_target":    0.05,    # Target air time per foot (seconds)
    "fac_pose":           0.8,     # Reward for staying near default pose
    "fac_feet_slip":      0.0,     # Penalize horizontal foot velocity during stance
    "fac_feet_clearance": 0.0,     # Penalize foot height deviation during swing
    "swing_height_target_front": 0.005,  # Target swing height for front legs (5mm)
    "swing_height_target_back": 0.025,   # Target swing height for back legs (25mm)

    # ── Termination ──
    "fallen_up_threshold": 0.0,    # Minimum up_z before considered fallen
}

# ── Training hyperparameters ─────────────────────────────────────────────────
LEARNING_RATE       = 3e-4
ENTROPY_COST        = 0.005
DISCOUNTING         = 0.97
NUM_STEPS           = 20_000_000
POLICY_LAYERS       = (128, 64, 32)
VALUE_LAYERS        = (128, 64, 32)
ACTIVATION          = "leaky_relu"
NUM_EVALS           = 5
EPISODE_LENGTH      = 250

# ── Misc ─────────────────────────────────────────────────────────────────────
DISCONNECT_WHEN_DONE = False

print("Configuration loaded (v4: Phase 0 — linear displacement, all penalties zeroed)")
print(f"  Training: {NUM_STEPS:,} steps, lr={LEARNING_RATE}, entropy={ENTROPY_COST}")
print(f"  Network:  policy={POLICY_LAYERS}, value={VALUE_LAYERS}, act={ACTIVATION}")
print(f"  Reward:   fac_tracking={REWARD_CONFIG['fac_tracking']}")
print(f"            All penalties at 0.0 for baseline measurement")
print(f"            Manual observation normalization is active (joint_vel, body_linvel, paw_vel, air_time)")


Configuration loaded (v4: Phase 0 — linear displacement, all penalties zeroed)
  Training: 20,000,000 steps, lr=0.0003, entropy=0.005
  Network:  policy=(128, 64, 32), value=(128, 64, 32), act=leaky_relu
  Reward:   fac_tracking=1000.0
            All penalties at 0.0 for baseline measurement
            Manual observation normalization is active (joint_vel, body_linvel, paw_vel, air_time)


In [4]:
mujoco_model = r'''<mujoco model="quadruped_scene">
  <compiler autolimits="true"/>
  <option timestep="0.004" iterations="4" ls_iterations="8"/>

  <asset>
    <material name="yellow" rgba="1 0.7 0 1" />
    <material name="black" rgba="0 0 0 1" />
    <material name="floor_gray" rgba="0.6 0.6 0.6 1" />
  </asset>

  <default>
    <joint damping="3.0e-2" armature="0.0005" frictionloss="2.0e-2" stiffness="0.0"/>
    <position kp="10.0" forcerange="-0.2 0.2"/>
    <geom contype="1" conaffinity="1" solimp="0.9 0.95 0.01" solref="0.005 1" margin="0.002" />

    <default class="upper_leg">
      <geom type="capsule" size="0.0046 0.0225" mass="0.005" material="black" euler="0 0 0"/>
    </default>

    <default class="lower_leg">
      <geom type="capsule" size="0.0046 0.025" mass="0.017" material="black" euler="0 90 0"/>
    </default>

    <default class="servo_upper_joint">
      <joint type="hinge" axis="0 1 0" pos="0 0 0.0225"/>
    </default>

    <default class="servo_lower_joint">
      <joint type="hinge" axis="0 1 0" pos="-0.025 0 0.0"/>
    </default>

    <default class="paw">
      <geom type="capsule" size="0.006 0.0025" pos="0.025 0 0" euler="90 0 0"
            mass="0.0005" material="black" friction="0.6 0.06 0.06" />
    </default>
  </default>

  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" material="floor_gray"/>

    <body name="body" pos="0 0 0.067">
      <geom type="box" size="0.0525 0.04 0.01" mass="0.111" material="yellow" />
      <freejoint/>
      <geom type="box" size="0.0525 0.04 0.01" material="black" rgba="0 0 0 0"/>

      <body name="battery" pos="0 0 -0.015">
        <geom type="box" size="0.045 0.02 0.0075" mass="0.068" material="black" />
      </body>

      <body name="left_upper_arm" pos="0.0525 0.05 -0.02">
        <geom class="upper_leg"/>
        <joint class="servo_upper_joint" name="joint_left_upper_arm"/>
        <body name="lowleg1" pos="0.023 -0.0175 -0.023">
          <geom class="lower_leg"/>
          <joint class="servo_lower_joint" name="joint_left_lower_arm"/>
          <geom class="paw" name="paw_LF"/>
          <site name="site_paw_LF" pos="0.025 0 0" size="0.002"/>
        </body>
      </body>

      <body name="right_upper_arm" pos="0.0525 -0.05 -0.02">
        <geom class="upper_leg" euler="0 0 0"/>
        <joint class="servo_upper_joint" name="joint_right_upper_arm"/>
        <body name="lowleg2" pos="0.023 0.0175 -0.023">
          <geom class="lower_leg"/>
          <joint class="servo_lower_joint" name="joint_right_lower_arm"/>
          <geom class="paw" name="paw_RF"/>
          <site name="site_paw_RF" pos="0.025 0 0" size="0.002"/>
        </body>
      </body>

      <body name="left_upper_leg" pos="-0.0525 0.05 -0.02">
        <geom class="upper_leg"/>
        <joint class="servo_upper_joint" name="joint_left_upper_leg"/>
        <body name="lowleg3" pos="0.023 -0.0175 -0.023">
          <geom class="lower_leg"/>
          <joint class="servo_lower_joint" name="joint_left_lower_leg"/>
          <geom class="paw" name="paw_LB"/>
          <site name="site_paw_LB" pos="0.025 0 0" size="0.002"/>
        </body>
      </body>

      <body name="right_upper_leg" pos="-0.0525 -0.05 -0.02">
        <geom class="upper_leg"/>
        <joint class="servo_upper_joint" name="joint_right_upper_leg"/>
        <body name="lowleg4" pos="0.023 0.0175 -0.023">
          <geom class="lower_leg"/>
          <joint class="servo_lower_joint" name="joint_right_lower_leg"/>
          <geom class="paw" name="paw_RB"/>
          <site name="site_paw_RB" pos="0.025 0 0" size="0.002"/>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name="l_up_arm" joint="joint_left_upper_arm"/>
    <position name="r_up_arm" joint="joint_right_upper_arm"/>
    <position name="r_up_leg" joint="joint_right_upper_leg"/>
    <position name="l_up_leg" joint="joint_left_upper_leg"/>
    <position name="l_low_arm" joint="joint_left_lower_arm"/>
    <position name="r_low_arm" joint="joint_right_lower_arm"/>
    <position name="r_low_leg" joint="joint_right_lower_leg"/>
    <position name="l_low_leg" joint="joint_left_lower_leg"/>
  </actuator>

  <sensor>
    <!-- Paw contact sensors: 1 if paw touches floor, 0 otherwise -->
    <touch name="touch_LF" site="site_paw_LF"/>
    <touch name="touch_RF" site="site_paw_RF"/>
    <touch name="touch_LB" site="site_paw_LB"/>
    <touch name="touch_RB" site="site_paw_RB"/>
    <!-- Paw velocity sensors -->
    <velocimeter name="vel_LF" site="site_paw_LF"/>
    <velocimeter name="vel_RF" site="site_paw_RF"/>
    <velocimeter name="vel_LB" site="site_paw_LB"/>
    <velocimeter name="vel_RB" site="site_paw_RB"/>
  </sensor>
</mujoco>
'''

In [5]:
import os
from typing import Dict, Any
import jax
import jax.numpy as jp
try:
    from brax.envs.base import Env, State
except ImportError:
    from brax.envs import env as _env
    Env = _env.Env
    State = _env.State
from brax.io import mjcf
from brax.mjx import pipeline

# Constants
EPISODE_LENGTH = 500
PERIOD = 0.02
LENGTH_JOINT_HISTORY = 30
LENGTH_RECENT_ANGLES = 3
SIZE_POLICY_OBS = 30  # body_quat(4) + body_angvel_scaled(2) + joint_angs_norm(8) + joint_vel(8) + last_action(8)
SIZE_PRIVILEGED_EXTRA = 3 + 4 + 8 + 4  # linvel(3) + paw_contacts(4) + paw_vel_xz(8) + feet_air_time(4)
SIZE_PRIVILEGED_OBS = SIZE_POLICY_OBS + SIZE_PRIVILEGED_EXTRA

# Limits and control
BOUND_ANG_DEG = 110
# Per-joint range [degrees] derived from IK gait model centers with safety margins
# [LF_shoulder, LF_elbow, RF_shoulder, RF_elbow, LB_hip, LB_knee, RB_hip, RB_knee]
# Shoulders/Hips (upper): ±40° (gait excursion ~36°), Elbows (lower front): ±20° (excursion ~14°), Knees (lower back): ±50° (excursion ~44°)
JOINT_RANGE_DEG = [40, 20, 40, 20, 40, 50, 40, 50]

# Reward factors (from PyBullet reference that produced walking gait)
ANG_FACTOR = 0.1              # Scale factor for angular velocity observations
JOINT_VEL_SCALE = 0.1         # Scale factor for joint velocity observations (servos are slow)

# Observation normalization (physical limits)
JOINT_VEL_MAX = 10.0      # rad/s, servo speed limit
BODY_LINVEL_MAX = 0.3     # m/s, max body velocity
PAW_VEL_MAX = 0.5         # m/s, max paw velocity
AIR_TIME_MAX = 0.4        # s, max expected swing time
FAC_Z_VELOCITY = 0.0          # Penalize vertical body velocity (disabled, same as PyBullet)
FAC_ARM_CONTACT = 0.0         # Penalize elbow contact (disabled — no contact sensor in MuJoCo model)
_POSE_WEIGHT = jp.array([1.0, 0.1, 1.0, 0.1, 1.0, 0.1, 1.0, 0.1])  # Upper joints=1.0, lower=0.1

_UP_VEC = jp.array([0.0, 0.0, 1.0])
_CONTACT_HEIGHT_THRESHOLD = 0.012


def _quat_rotate(q, v):
    """Rotate vector v by quaternion q=(w,x,y,z)."""
    w, x, y, z = q
    qvec = jp.array([x, y, z])
    t = 2.0 * jp.cross(qvec, v)
    return v + w * t + jp.cross(qvec, t)


def _quat_inv(q):
    """Inverse of unit quaternion q=(w,x,y,z)."""
    return jp.array([q[0], -q[1], -q[2], -q[3]])


def _quat_multiply(q1, q2):
    """Multiply two quaternions q1*q2, both in (w,x,y,z) format."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return jp.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ])


class OpenCatBittleBraxEnv(Env):
    """Brax/MJX quadruped with asymmetric actor-critic (privileged state for value)."""

    def __init__(self, xml_path: str = None, **reward_overrides):
        # Merge defaults from REWARD_CONFIG with any overrides
        _cfg = {**REWARD_CONFIG, **reward_overrides}
        if xml_path is None:
            xml_path = os.path.join(os.path.dirname(__file__), "models", "opencat_environment.xml")
        self.sys = mjcf.loads(xml_path)
        self._bound_ang = jp.deg2rad(BOUND_ANG_DEG)
        self._joint_range = jp.deg2rad(jp.array(JOINT_RANGE_DEG))
        self._fac_tracking = _cfg["fac_tracking"]
        self._fac_smooth_1 = _cfg["fac_smooth_1"]
        self._fac_smooth_2 = _cfg["fac_smooth_2"]
        self._fac_stability = _cfg["fac_stability"]
        self._fac_orient_pitch = _cfg["fac_orient_pitch"]
        self._fac_orient_roll = _cfg["fac_orient_roll"]
        self._fac_lin_vel_z = _cfg.get("fac_lin_vel_z", 0.0)
        self._fac_air_time = _cfg["fac_air_time"]
        self._air_time_target = _cfg["air_time_target"]
        self._fac_pose = _cfg["fac_pose"]
        self._fallen_up_threshold = _cfg["fallen_up_threshold"]
        self._fac_feet_slip = _cfg["fac_feet_slip"]
        self._fac_feet_clearance = _cfg["fac_feet_clearance"]
        self._swing_height_target_front = _cfg.get("swing_height_target_front", 0.005)
        self._swing_height_target_back = _cfg.get("swing_height_target_back", 0.025)

        # Initial pose: default stance from IK gait model [51, -5, 51, -5, 72, 3, 72, 3] degrees
        # Joint order: [LF_shoulder, LF_elbow, RF_shoulder, RF_elbow, LB_hip, LB_knee, RB_hip, RB_knee]
        self._joint_angs_init = jp.deg2rad(
            jp.array([51.0, -5.0, 51.0, -5.0, 72.0, 3.0, 72.0, 3.0])
        ) / self._bound_ang

        self._hold_steps = max(1, int(round(PERIOD / float(self.sys.opt.timestep))))
        self._body_id = _find_body_id(self.sys, "body")
        self._joint_qpos_start = 7
        self._actuator_from_joint = jp.array([0, 2, 6, 4, 1, 3, 7, 5])
        self._paw_site_ids = jp.array([0, 1, 2, 3])

        # Precompute default angles in radians for normalization
        self._default_angles = self._joint_angs_init * self._bound_ang

    @property
    def observation_size(self) -> Dict[str, int]:
        return {"state": SIZE_POLICY_OBS, "privileged_state": SIZE_PRIVILEGED_OBS}

    @property
    def action_size(self) -> int:
        return int(self.sys.act_size())

    @property
    def backend(self) -> str:
        return "mjx"

    @property
    def dt(self) -> float:
        return PERIOD

    def reset(self, rng: jp.ndarray) -> State:
        rng, rng1, rng2, rng3 = jax.random.split(rng, 4)

        q = jp.array(self.sys.qpos0)
        qd = jp.zeros(self.sys.qd_size())

        # Randomize joint angles: default pose + uniform noise ±0.05 rad (~3°)
        init_joint_angles = self._joint_angs_init * self._bound_ang
        joint_noise = jax.random.uniform(rng1, (self.action_size,), minval=-0.05, maxval=0.05)
        init_joint_angles = init_joint_angles + joint_noise
        q = q.at[self._joint_qpos_start:self._joint_qpos_start + self.action_size].set(init_joint_angles)

        # Randomize base velocity: ±0.1 m/s translational, ±0.2 rad/s rotational
        base_vel_noise = jax.random.uniform(rng2, (6,), minval=jp.array([-0.1, -0.1, -0.05, -0.2, -0.2, -0.2]),
                                                      maxval=jp.array([0.1, 0.1, 0.05, 0.2, 0.2, 0.2]))
        qd = qd.at[:6].set(base_vel_noise)

        # Randomize base orientation: small roll/pitch perturbation ±0.05 rad (~3°)
        roll_pitch_noise = jax.random.uniform(rng3, (2,), minval=-0.05, maxval=0.05)
        # Apply small rotation to quaternion (q[3:7] is wxyz quaternion)
        # For small angles: q_perturbed ≈ q_original * [1, roll/2, pitch/2, 0]
        dq = jp.array([1.0, roll_pitch_noise[0] / 2.0, roll_pitch_noise[1] / 2.0, 0.0])
        dq = dq / jp.linalg.norm(dq)  # normalize
        orig_quat = q[3:7]
        # Quaternion multiply: result = orig * dq
        new_quat = _quat_multiply(orig_quat, dq)
        q = q.at[3:7].set(new_quat)

        sim_state = pipeline.init(self.sys, q, qd)

        # NO settling loop — start immediately (like PyBullet reference)

        # Compute initial joint_angs_norm from the actual initial joint angles (after noise)
        init_joint_angs_norm = (init_joint_angles - self._default_angles) / self._joint_range

        recent_angles = jp.zeros(LENGTH_RECENT_ANGLES * 8)
        last_action = jp.zeros(self.action_size)
        obs = self._get_obs(sim_state, init_joint_angs_norm, last_action)

        info = {
            "recent_angles": recent_angles,
            "step_counter": jp.array(0, dtype=jp.int32),
            "truncation": jp.array(0.0, dtype=jp.float32),
            "last_action": last_action,
            "time_out": jp.array(0.0, dtype=jp.float32),
            "air_time": jp.zeros(4),
            "first_contact": jp.zeros(4),
            "prev_paw_pos": sim_state.site_xpos[self._paw_site_ids],
            "forward_displacement": jp.array(0.0),
        }
        metrics = {
            "reward": jp.array(0.0),
            "forward": jp.array(0.0),       # FAC_MOVEMENT * dx
            "smooth_1": jp.array(0.0),
            "smooth_2": jp.array(0.0),
            "stability": jp.array(0.0),     # Angular velocity penalty
            "orientation": jp.array(0.0),   # Orientation penalty
            "lin_vel_z": jp.array(0.0),     # Vertical velocity penalty
            "forward_vel": jp.array(0.0),   # Diagnostic: instantaneous forward velocity
            "up_z": jp.array(1.0),          # Diagnostic: upright-ness
            "fallen": jp.array(0.0),        # Diagnostic: fall flag
            "air_time": jp.array(0.0),
            "pose": jp.array(0.0),
            "feet_slip": jp.array(0.0),
            "feet_clearance": jp.array(0.0),
        }
        return State(sim_state, obs, jp.array(0.0), jp.array(0.0), metrics, info)

    def step(self, state: State, action: jp.ndarray) -> State:
        sim_state = state.pipeline_state
        # Absolute target: action [-1, 1] maps to default_pose ± JOINT_RANGE
        default_angles = self._default_angles
        joint_angs = default_angles + action * self._joint_range
        joint_angs = jp.clip(joint_angs, -self._bound_ang, self._bound_ang)
        actuator_targets = joint_angs[self._actuator_from_joint]

        def body(s, _):
            return pipeline.step(self.sys, s, actuator_targets), None
        sim_state, _ = jax.lax.scan(body, sim_state, None, length=self._hold_steps)

        # Compute normalized joint angles for observation and smoothness penalties
        joint_angs_norm = (joint_angs - self._default_angles) / self._joint_range

        # Update recent angles buffer (rolling 3-frame window for smoothness penalties)
        recent_angles = jp.concatenate((state.info["recent_angles"][8:], joint_angs_norm), axis=0)
        # Layout: [prev_prev (0:8), prev (8:16), current (16:24)]
        joint_angs_prev = recent_angles[8:16]
        joint_angs_prev_prev = recent_angles[0:8]

        # ========== REWARD: Forward displacement (linear, like PyBullet reference) ==========

        # 0. Body orientation (needed for uprightness gate and orientation penalty)
        body_quat = sim_state.xquat[self._body_id]
        up_vec = _quat_rotate(body_quat, _UP_VEC)

        # 1. Forward displacement reward (linear, like PyBullet reference)
        body_x_now = sim_state.xpos[self._body_id, 0]
        body_x_prev = state.pipeline_state.xpos[self._body_id, 0]
        forward_displacement = body_x_now - body_x_prev

        reward_forward = self._fac_tracking * forward_displacement

        # Diagnostic: local forward velocity (not used in reward)
        global_linvel = sim_state.qd[:3]
        local_linvel = _quat_rotate(_quat_inv(body_quat), global_linvel)
        forward_vel = local_linvel[0]

        # 2. Smoothness penalty (1st and 2nd order, from PyBullet reference)
        smooth_1 = jp.sum(self._fac_smooth_1 * jp.square(joint_angs_norm - joint_angs_prev))
        smooth_2 = jp.sum(self._fac_smooth_2 * jp.square(
            joint_angs_norm - 2.0 * joint_angs_prev + joint_angs_prev_prev
        ))
        penalty_smooth = smooth_1 + smooth_2

        # 3. Body stability penalty (angular velocity roll and pitch, using scaled values)
        body_angvel = sim_state.cvel[self._body_id, :3]  # [wx, wy, wz] in body frame
        body_angvel_scaled = jp.clip(body_angvel[:2] * ANG_FACTOR, -1.0, 1.0)
        penalty_stability = self._fac_stability * jp.sum(jp.square(body_angvel_scaled))

        # 4. Orientation penalty (always active, not gated by penalty_scale — playground style)
        penalty_orientation = self._fac_orient_pitch * up_vec[0]**2 + self._fac_orient_roll * up_vec[1]**2

        # 4b. Vertical velocity penalty (anti-bounce)
        lin_vel_z = global_linvel[2]
        penalty_lin_vel_z = self._fac_lin_vel_z * jp.square(lin_vel_z)

        # 5. Pose reward (always active — playground style, Gaussian centered on default pose)
        reward_pose = self._fac_pose * jp.exp(-jp.sum(jp.square(joint_angs - self._default_angles) * _POSE_WEIGHT))

        # 6. Feet air time (gated by penalty_scale)
        contact = self._get_paw_contacts(sim_state)
        first_contact = jp.maximum(state.info["first_contact"], contact)
        air_time = state.info["air_time"] + self.dt
        air_time = air_time * (1.0 - contact)  # Reset on contact
        reward_air_time = self._fac_air_time * jp.sum((air_time - self._air_time_target) * first_contact)

        # 7. Feet slip penalty — penalize horizontal foot velocity during stance
        prev_paw_pos = state.info["prev_paw_pos"]
        curr_paw_pos = sim_state.site_xpos[self._paw_site_ids]  # (4, 3)
        paw_vel = (curr_paw_pos - prev_paw_pos) / self.dt
        paw_vel_xy = paw_vel[:, :2]  # Only horizontal components
        vel_xy_norm_sq = jp.sum(jp.square(paw_vel_xy), axis=-1)  # (4,)
        feet_slip = jp.sum(vel_xy_norm_sq * contact)  # Only penalize during stance

        # 8. Feet clearance — penalize foot height deviation from target during swing
        # Paw order: [LF, RF, LB, RB] (indices 0,1 are front; indices 2,3 are back)
        paw_z = sim_state.site_xpos[self._paw_site_ids, 2]  # (4,)
        paw_vel_norm = jp.sqrt(jp.sum(jp.square(paw_vel_xy), axis=-1))  # (4,)
        swing_targets = jp.array([self._swing_height_target_front, self._swing_height_target_front,
                                  self._swing_height_target_back, self._swing_height_target_back])
        height_delta = jp.abs(paw_z - swing_targets)  # (4,)
        feet_clearance = jp.sum(height_delta * paw_vel_norm)  # Only active when foot is moving

        # 9. Total reward: tracking + pose + air_time - all penalties
        reward_raw = (reward_forward
                      + reward_pose
                      + reward_air_time
                      - penalty_smooth
                      - penalty_stability
                      - penalty_orientation
                      - penalty_lin_vel_z
                      - self._fac_feet_slip * feet_slip
                      - self._fac_feet_clearance * feet_clearance)

        # ========== TERMINATION ==========
        fallen = up_vec[2] < self._fallen_up_threshold

        step_counter = state.info["step_counter"] + 1
        truncated = step_counter >= EPISODE_LENGTH
        done = jp.maximum(fallen.astype(jp.float32), truncated.astype(jp.float32))

        # On fall: reward = 0.0 (termination penalty)
        reward_raw = jp.where(fallen, -1.0, reward_raw)

        # Multiply by dt (allow negative per-step reward for backward motion)
        reward = reward_raw * self.dt

        # ========== OBSERVATION ==========
        obs = self._get_obs(sim_state, joint_angs_norm, action, air_time=air_time, prev_paw_pos=state.info["prev_paw_pos"])

        # ========== UPDATE STATE ==========
        new_info = dict(state.info)
        new_info.update({
            "recent_angles": recent_angles,
            "step_counter": step_counter,
            "truncation": jp.where(fallen, 0.0, truncated.astype(jp.float32)),
            "last_action": action,
            "time_out": jp.where(fallen, 0.0, truncated.astype(jp.float32)),
            "air_time": air_time,
            "first_contact": first_contact,
            "prev_paw_pos": curr_paw_pos,
            "forward_displacement": forward_displacement,
        })

        new_metrics = dict(state.metrics)
        new_metrics.update({
            "reward": reward,
            "forward": reward_forward * self.dt,     # Linear displacement reward component
            "smooth_1": -smooth_1 * self.dt,
            "smooth_2": -smooth_2 * self.dt,
            "stability": -penalty_stability * self.dt, # Negative = penalty (for display)
            "orientation": -penalty_orientation * self.dt, # Negative = penalty
            "lin_vel_z": -penalty_lin_vel_z * self.dt, # Negative = penalty
            "forward_vel": local_linvel[0],           # Diagnostic
            "up_z": up_vec[2],                        # Diagnostic
            "fallen": fallen.astype(jp.float32),      # Diagnostic
            "air_time": reward_air_time * self.dt,
            "pose": reward_pose * self.dt,
            "feet_slip": -feet_slip * self._fac_feet_slip * self.dt,
            "feet_clearance": -feet_clearance * self._fac_feet_clearance * self.dt,
        })

        return state.replace(
            pipeline_state=sim_state, obs=obs, reward=reward, done=done,
            info=new_info, metrics=new_metrics
        )

    def _get_obs(self, sim_state, joint_angs_norm: jp.ndarray, last_action: jp.ndarray, air_time: jp.ndarray = None, prev_paw_pos: jp.ndarray = None) -> Dict[str, jp.ndarray]:
        """Returns policy and privileged observations.

        Policy observation (30 dims):
        - body_quat (4): full quaternion from sim_state.xquat[self._body_id]
        - body_angvel_scaled (2): scaled angular velocity roll+pitch (rad/s)
        - joint_angs_norm (8): normalized joint angles
        - joint_vel (8): joint velocities from sim_state.qd[6:6 + self.action_size]
        - last_action (8)

        Privileged observation adds (on top of policy obs):
        - body_linvel (3): body linear velocity
        - paw_contacts (4): paw contact flags
        - paw_vel_xz (8): paw velocities in x and z directions (2 per paw)
        - feet_air_time (4): time each foot has been in the air
        Total: 49
        """
        # Default handling
        if air_time is None:
            air_time = jp.zeros(4)
        if prev_paw_pos is None:
            prev_paw_pos = sim_state.site_xpos[self._paw_site_ids]

        # Policy observation components
        body_quat = sim_state.xquat[self._body_id]  # (4,)

        # Body angular velocity: roll+pitch only, scaled and clipped
        body_angvel = sim_state.cvel[self._body_id, :3]  # [wx, wy, wz] in body frame
        body_angvel_scaled = jp.clip(body_angvel[:2] * ANG_FACTOR, -1.0, 1.0)  # Scale and clip angular velocity

        # Joint velocities
        joint_vel = sim_state.qd[6:6 + self.action_size]  # (8,)

        # Normalize joint velocities
        joint_vel_norm = jp.clip(joint_vel / JOINT_VEL_MAX, -1.0, 1.0)  # (8,)

        # Construct policy observation: body_quat + body_angvel_scaled + joint_angs_norm + joint_vel + last_action
        policy_obs = jp.concatenate([
            body_quat,           # 4
            body_angvel_scaled,  # 2
            joint_angs_norm,     # 8
            joint_vel_norm,           # 8
            last_action,         # 8
        ])  # Total: 30

        # Privileged observation extras
        body_linvel = sim_state.cvel[self._body_id, 3:6]  # (3,)
        paw_contacts = self._get_paw_contacts(sim_state)  # (4,)

        # Paw velocities (x and z components only, for each of 4 paws)
        curr_paw_pos = sim_state.site_xpos[self._paw_site_ids]  # (4, 3)
        paw_vel = (curr_paw_pos - prev_paw_pos) / PERIOD  # (4, 3)
        paw_vel_xz = jp.concatenate([paw_vel[:, 0:1], paw_vel[:, 2:3]], axis=-1).flatten()  # (8,) - x,z for each paw

        # Normalize privileged observations
        body_linvel_norm = jp.clip(body_linvel / BODY_LINVEL_MAX, -1.0, 1.0)  # (3,)
        paw_vel_xz_norm = jp.clip(paw_vel_xz / PAW_VEL_MAX, -1.0, 1.0)  # (8,)
        air_time_norm = jp.clip(air_time / AIR_TIME_MAX, -1.0, 1.0)  # (4,)

        privileged_obs = jp.concatenate([
            policy_obs,         # 30
            body_linvel_norm,        # 3
            paw_contacts,       # 4
            paw_vel_xz_norm,         # 8
            air_time_norm,           # 4
        ])  # Total: 49

        return {"state": policy_obs, "privileged_state": privileged_obs}

    def _get_actuators(self, sim_state) -> jp.ndarray:
        return sim_state.q[self._joint_qpos_start:self._joint_qpos_start + self.action_size]

    def _get_paw_contacts(self, sim_state) -> jp.ndarray:
        paw_z = sim_state.site_xpos[self._paw_site_ids, 2]
        return (paw_z < _CONTACT_HEIGHT_THRESHOLD).astype(jp.float32)


def _find_body_id(sys, name: str) -> int:
    """Find body ID by name."""
    if hasattr(sys, "link_names"):
        return list(sys.link_names).index(name) + 1
    if hasattr(sys, "body_names"):
        return list(sys.body_names).index(name) + 1
    if hasattr(sys, "body"):
        return [b.name for b in sys.body].index(name) + 1
    raise ValueError(f"Unable to find body '{name}' in system")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [6]:
import warnings
warnings.filterwarnings("ignore", message=".*Brax System.*not actively being maintained.*")
import flax.linen as nn
from brax.training.agents.ppo import train as ppo_lib
from brax.training.agents.ppo import networks as ppo_networks

_activations = {"leaky_relu": nn.leaky_relu, "swish": nn.swish, "relu": nn.relu, "tanh": nn.tanh}
act_fn = _activations.get(ACTIVATION, nn.leaky_relu)

env = OpenCatBittleBraxEnv(mujoco_model, **REWARD_CONFIG)

eval_metrics_history = {}
header_printed = False
metric_order = [
    "reward", "forward", "air_time", "pose", "smooth_1", "smooth_2",
    "stability", "orientation", "lin_vel_z", "forward_vel", "up_z", "fallen",
    "feet_slip", "feet_clearance",
]
metric_display = {
    "reward": "reward", "forward": "fwd_dx", "air_time": "air_t",
    "pose": "pose", "smooth_1": "smth_1", "smooth_2": "smth_2",
    "stability": "stabil", "orientation": "orient", "lin_vel_z": "lvz",
    "forward_vel": "fwd_v", "up_z": "up_z", "fallen": "fall",
    "feet_slip": "slip", "feet_clearance": "clear",
}

def _progress(num_steps, metrics):
    global header_printed
    if num_steps <= 0:
        return
    for key, val in metrics.items():
        if not key.endswith("_std"):
            if key not in eval_metrics_history:
                eval_metrics_history[key] = []
            eval_metrics_history[key].append((num_steps, float(val)))
    if not header_printed:
        print(f"{'Step':>9s}" + "".join(f"{metric_display.get(m, m):>9s}" for m in metric_order))
        header_printed = True
    row = f"{int(num_steps):>9d}"
    for name in metric_order:
        key = f"eval/episode_{name}"
        row += f"{metrics[key]:>9.3f}" if key in metrics else f"{'N/A':>9s}"
    print(row)

_nf = lambda obs_size, act_size, preprocess_observations_fn=None: (
    ppo_networks.make_ppo_networks(
        obs_size, act_size,
        preprocess_observations_fn=preprocess_observations_fn,
        policy_hidden_layer_sizes=POLICY_LAYERS,
        value_hidden_layer_sizes=VALUE_LAYERS,
        policy_obs_key="state", value_obs_key="privileged_state",
        activation=act_fn))

print(f"Training {NUM_STEPS:,} steps...")
print(f"Reward config: {REWARD_CONFIG}")
make_inference_fn, params, _ = ppo_lib.train(
    environment=env,
    num_timesteps=NUM_STEPS,
    episode_length=EPISODE_LENGTH,
    action_repeat=1,
    num_envs=8192,
    unroll_length=20,#64,
    num_minibatches=32,
    num_updates_per_batch=4,
    batch_size=256,
    clipping_epsilon=0.2,
    network_factory=_nf,
    learning_rate=LEARNING_RATE,
    entropy_cost=ENTROPY_COST,
    discounting=DISCOUNTING,
    normalize_observations=False,
    num_evals=NUM_EVALS,
    progress_fn=_progress,
    log_training_metrics=False,
    max_grad_norm=1.0,
    bootstrap_on_timeout=True,
)

print("\nTraining complete.")
print(f"\nREWARD CONFIG")
print(f"{'Key':<20s} {'Value':>10s}")
for key, val in REWARD_CONFIG.items():
    print(f"{key:<20s} {val:10.4f}")

Training 20,000,000 steps...
Reward config: {'fac_tracking': 1000.0, 'fac_smooth_1': 0.0, 'fac_smooth_2': 0.0, 'fac_stability': 0.0, 'fac_orient_pitch': 0.0, 'fac_orient_roll': 0.0, 'fac_lin_vel_z': 0.0, 'fac_air_time': 0.0, 'air_time_target': 0.05, 'fac_pose': 0.8, 'fac_feet_slip': 0.0, 'fac_feet_clearance': 0.0, 'swing_height_target_front': 0.005, 'swing_height_target_back': 0.025, 'fallen_up_threshold': 0.0}
     Step   reward   fwd_dx    air_t     pose   smth_1   smth_2   stabil   orient      lvz    fwd_v     up_z     fall     slip    clear
  5079040   35.621   31.297    0.000    4.327    0.000    0.000    0.000    0.000    0.000   80.676  461.627    0.125    0.000    0.000
 10158080   29.498   25.014    0.000    4.481    0.000    0.000    0.000    0.000    0.000   67.486  406.261    0.273    0.000    0.000
 15237120   41.171   35.796    0.000    5.373    0.000    0.000    0.000    0.000    0.000   92.509  465.883    0.109    0.000    0.000
 20316160   40.621   35.414    0.000    5

In [7]:
# ── Save policy params for deployment ──
import numpy as np
import pickle

# Extract policy weights from params tuple
# params is (normalizer_params, policy_params, value_params)
# We only need policy_params = params[1]
policy_params = params[1]

# Build weights dictionary from the policy network structure
weights = {}

# Extract weights from each hidden layer
for layer_name in ['hidden_0', 'hidden_1', 'hidden_2', 'hidden_3']:
    kernel = np.array(policy_params['params'][layer_name]['kernel'])
    bias = np.array(policy_params['params'][layer_name]['bias'])

    weights[f'{layer_name}_kernel'] = kernel
    weights[f'{layer_name}_bias'] = bias

# Add config for verification
weights['_config_action_size'] = np.array([env.action_size])
weights['_config_policy_obs_size'] = np.array([30])

# Save to NPZ file
np.savez('bittle_policy_weights.npz', **weights)

print("Saved policy weights to bittle_policy_weights.npz")
print(f"  File contains: {len(weights)} arrays")
print("\nWeight shapes (layer_kernel and layer_bias):")
for layer_name in ['hidden_0', 'hidden_1', 'hidden_2', 'hidden_3']:
    kernel = weights[f'{layer_name}_kernel']
    bias = weights[f'{layer_name}_bias']
    print(f"  {layer_name}_kernel: {kernel.shape}")
    print(f"  {layer_name}_bias:   {bias.shape}")

print(f"\nConfig:")
print(f"  _config_action_size: {weights['_config_action_size']}")
print(f"  _config_policy_obs_size: {weights['_config_policy_obs_size']}")

# ── Commented out: old pickle export (kept for reference) ──
# with open("bittle_policy_params.pkl", "wb") as f:
#     pickle.dump(params, f)
#
# config = {
#     "policy_layers": POLICY_LAYERS,
#     "value_layers": VALUE_LAYERS,
#     "activation": ACTIVATION,
#     "obs_size": env.observation_size,
#     "action_size": env.action_size,
# }
# with open("bittle_policy_config.pkl", "wb") as f:
#     pickle.dump(config, f)
#
# print(f"Saved params and config for deployment.")
# print(f"  Policy obs: {config['obs_size']['state']} dims")
# print(f"  Action: {config['action_size']} dims")
# print(f"  Network: policy={config['policy_layers']}, value={config['value_layers']}")
# print(f"Download: bittle_policy_params.pkl + bittle_policy_config.pkl")

Saved policy weights to bittle_policy_weights.npz
  File contains: 10 arrays

Weight shapes (layer_kernel and layer_bias):
  hidden_0_kernel: (30, 128)
  hidden_0_bias:   (128,)
  hidden_1_kernel: (128, 64)
  hidden_1_bias:   (64,)
  hidden_2_kernel: (64, 32)
  hidden_2_bias:   (32,)
  hidden_3_kernel: (32, 16)
  hidden_3_bias:   (16,)

Config:
  _config_action_size: [8]
  _config_policy_obs_size: [30]


In [8]:
# This cell has been merged with Cell 8 (visualization + gait assessment)
# See Cell 8 for the merged code.

In [9]:
from brax.io import html
from IPython.display import HTML, display
import jax
import jax.numpy as jp
import numpy as np

inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

rng = jax.random.PRNGKey(0)
state = env.reset(rng)

def rollout(state, rng):
    def step_fn(carry, _):
        state, rng = carry
        rng, act_key = jax.random.split(rng)
        action, _ = jit_inference_fn(state.obs, act_key)
        state = env.step(state, action)
        return (state, rng), state.pipeline_state
    (state, rng), traj = jax.lax.scan(step_fn, (state, rng), None, length=EPISODE_LENGTH)
    return state, traj

reset_pipeline_state = state.pipeline_state
jit_rollout = jax.jit(rollout)
state, traj = jit_rollout(state, rng)

trajectory = [reset_pipeline_state]
trajectory.extend([
    jax.tree_util.tree_map(lambda x, i=i: x[i], traj)
    for i in range(traj.q.shape[0])
])

control_dt = float(env.sys.opt.timestep) * env._hold_steps
render_sys = env.sys.replace(opt=env.sys.opt.replace(timestep=control_dt))
html_str = html.render(render_sys, trajectory, colab=True)
display(HTML(html_str))

# ── Gait Assessment (from same rollout) ──
# Extract kinematics from trajectory
body_pos_all = traj.xpos[:, env._body_id]  # (T, 3)
body_x = jax.device_get(body_pos_all[:, 0])
body_y = jax.device_get(body_pos_all[:, 1])
body_z = jax.device_get(body_pos_all[:, 2])

body_quat_all = traj.xquat[:, env._body_id]  # (T, 4)
# Compute up_z from quaternion (MuJoCo format: w, x, y, z)
def _batch_quat_up_z(quats):
    """Extract z-component of up vector from quaternions (w,x,y,z)."""
    w, x, y, z = quats[:, 0], quats[:, 1], quats[:, 2], quats[:, 3]
    # up_z = 1 - 2*(x^2 + y^2) for unit quaternion
    return 1.0 - 2.0 * (x**2 + y**2)

up_z = jax.device_get(_batch_quat_up_z(body_quat_all))

paw_z_all = traj.site_xpos[:, env._paw_site_ids, 2]  # (T, 4)
paw_z_np = jax.device_get(paw_z_all)
contacts = (paw_z_np < _CONTACT_HEIGHT_THRESHOLD).astype(float)

# Now run the gait assessment analysis
net_forward = float(body_x[-1] - body_x[0])
net_lateral = float(body_y[-1] - body_y[0])
paw_names = ["LF", "RF", "LB", "RB"]

print(f"Forward displacement: {net_forward*100:.2f} cm")
print(f"Lateral displacement: {net_lateral*100:.2f} cm")
print(f"Avg height: {float(body_z.mean()):.4f} m")
print(f"Avg up_z: {float(up_z.mean()):.3f}")
print()

total_steps = 0
dt = float(env.dt)
for i, name in enumerate(paw_names):
    c = contacts[:, i]
    duty = float(c.mean())
    transitions = c[1:] - c[:-1]
    lift_offs = int((transitions < -0.5).sum())
    total_steps += lift_offs
    in_air = (c < 0.5)
    air_durs = []
    cur = 0
    for t in range(len(c)):
        if in_air[t]:
            cur += 1
        else:
            if cur > 0:
                air_durs.append(cur * dt)
            cur = 0
    if cur > 0:
        air_durs.append(cur * dt)
    avg_air = sum(air_durs) / len(air_durs) if air_durs else 0.0
    print(f"  {name}: {lift_offs:>3d} steps, duty={duty:.2f}, avg_air={avg_air:.3f}s")

print(f"\nTotal steps: {total_steps}")

lf_rb_sync = float((contacts[:, 0] == contacts[:, 3]).mean())
rf_lb_sync = float((contacts[:, 1] == contacts[:, 2]).mean())
lf_rf_anti = float((contacts[:, 0] != contacts[:, 1]).mean())
print(f"Diagonal sync: LF-RB={lf_rb_sync:.3f}, RF-LB={rf_lb_sync:.3f}")
print(f"LF-RF antiphase: {lf_rf_anti:.3f}")
print(f"  (trotting: sync > 0.7, antiphase > 0.5)")

# Qualitative assessment
if total_steps == 0:
    gait = "NO_STEPPING"
elif net_forward < 0.005:
    gait = f"STEPPING_IN_PLACE: {total_steps} steps but < 5mm forward"
elif lf_rb_sync > 0.7 and rf_lb_sync > 0.7 and lf_rf_anti > 0.4:
    gait = f"TROTTING: {total_steps} steps, {net_forward*100:.1f}cm forward"
elif total_steps > 8:
    gait = f"WALKING: {total_steps} steps, {net_forward*100:.1f}cm forward"
else:
    gait = f"ATTEMPTING: {total_steps} steps, {net_forward*100:.1f}cm forward"
print(f"\nGait: {gait}")

# ── Raw values for penalty calibration ──
# Extract raw physics values to help calibrate penalty magnitudes
joint_angs_all = traj.q[:, env._joint_qpos_start:env._joint_qpos_start + env.action_size]
joint_angs_np = jax.device_get(joint_angs_all)

# Vertical velocity
body_vel_all = traj.qd[:, :3]  # (T, 3) global linear velocity
body_vel_np = jax.device_get(body_vel_all)
lin_vel_z_values = body_vel_np[:, 2]

# Foot slip velocities during stance
paw_pos_all = jax.device_get(traj.site_xpos[:, env._paw_site_ids])  # (T, 4, 3)
paw_vel_all = np.diff(paw_pos_all, axis=0) / dt  # (T-1, 4, 3)
paw_vel_xy_sq = np.sum(paw_vel_all[:, :, :2]**2, axis=-1)  # (T-1, 4)
stance_mask = contacts[1:, :]  # align with velocity
slip_during_stance = paw_vel_xy_sq * stance_mask

print(f"\n── Raw values for penalty calibration ──")
print(f"  lin_vel_z:  mean={np.mean(np.abs(lin_vel_z_values)):.4f}, max={np.max(np.abs(lin_vel_z_values)):.4f} m/s")
print(f"  feet_slip:  mean={np.mean(slip_during_stance):.6f}, max={np.max(slip_during_stance):.6f} m²/s²")
print(f"  paw_z (swing): ", end="")
for i, name in enumerate(paw_names):
    swing_mask = contacts[:, i] < 0.5
    if swing_mask.any():
        pz = paw_z_np[swing_mask, i]
        print(f"{name}={np.mean(pz):.4f}m ", end="")
    else:
        print(f"{name}=no_swing ", end="")
print()
print(f"  joint_range: min={np.min(np.degrees(joint_angs_np)):.1f}°, max={np.max(np.degrees(joint_angs_np)):.1f}°")
print(f"  forward_vel: mean={np.mean(body_vel_np[:, 0]):.4f}, max={np.max(body_vel_np[:, 0]):.4f} m/s")

Forward displacement: 126.43 cm
Lateral displacement: -23.82 cm
Avg height: 0.0503 m
Avg up_z: 0.285

  LF:  24 steps, duty=0.34, avg_air=0.274s
  RF:  15 steps, duty=0.40, avg_air=0.400s
  LB:  24 steps, duty=0.28, avg_air=0.302s
  RB:  23 steps, duty=0.47, avg_air=0.231s

Total steps: 86
Diagonal sync: LF-RB=0.658, RF-LB=0.632
LF-RF antiphase: 0.510
  (trotting: sync > 0.7, antiphase > 0.5)

Gait: WALKING: 86 steps, 126.4cm forward

── Raw values for penalty calibration ──
  lin_vel_z:  mean=0.0498, max=0.5957 m/s
  feet_slip:  mean=0.016787, max=0.776864 m²/s²
  paw_z (swing): LF=0.0545m RF=0.0489m LB=0.0363m RB=0.0588m 
  joint_range: min=-45.9°, max=95.2°
  forward_vel: mean=0.1269, max=0.4159 m/s


In [10]:
# Disconnect from runtime to save compute units.
# Controlled by DISCONNECT_WHEN_DONE in the config cell.
if DISCONNECT_WHEN_DONE:
    print("DISCONNECT_WHEN_DONE = True — releasing runtime now.")
    from google.colab import runtime
    runtime.unassign()
else:
    print("DISCONNECT_WHEN_DONE = False — runtime kept alive.")
    print("Disconnect manually via Runtime > Disconnect and delete runtime.")

DISCONNECT_WHEN_DONE = False — runtime kept alive.
Disconnect manually via Runtime > Disconnect and delete runtime.
